# Números combinatorios 🔢
### Una introducción práctica para estudiantes de primer año de CS

En este notebook vamos a entender **qué son los números combinatorios** (coeficientes binomiales),
**de dónde salen**, y **cómo calcularlos en Python** de varias formas distintas, comparando su eficiencia.

**Índice**
1. El problema: contar sin enumerar
2. El principio de multiplicación
3. Permutaciones
4. Variaciones
5. Combinaciones y el número combinatorio
6. La fórmula
7. Implementación en Python (varias versiones)
8. El triángulo de Pascal
9. Por qué importa en Computer Science
10. Ejercicios para practicar

## 1. El problema: contar sin enumerar

Imagina que tienes que elegir un equipo de **3 personas** de un grupo de **10**. ¿Cuántos equipos distintos puedes formar?

Podrías escribir todos los equipos a mano... pero con 10 personas ya hay **120** combinaciones posibles.
Con 30 personas, ¡hay más de 4 millones! Necesitamos una forma de **contar sin enumerar uno por uno**.

Eso es exactamente lo que resuelve la **combinatoria**, y el número combinatorio es una de sus herramientas más importantes.

In [ ]:
# Vamos a comprobarlo por fuerza bruta con Python, para grupos pequeños
from itertools import combinations

personas = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']  # 10 personas
equipos = list(combinations(personas, 3))

print(f'Número de equipos posibles: {len(equipos)}')
print('Algunos ejemplos:', equipos[:5])

## 2. El principio de multiplicación

Antes de llegar a las combinaciones, necesitamos la idea base de toda la combinatoria:

> Si una tarea se puede descomponer en pasos, y el paso 1 tiene $n_1$ opciones, el paso 2 tiene $n_2$ opciones, etc.,
> el número total de formas de hacer la tarea completa es $n_1 \times n_2 \times \dots \times n_k$.

**Ejemplo:** si tienes 3 camisetas y 4 pantalones, tienes $3 \times 4 = 12$ combinaciones de ropa distintas.

In [ ]:
camisetas = ['roja', 'azul', 'verde']
pantalones = ['negro', 'blanco', 'gris', 'beige']

combinaciones_ropa = [(c, p) for c in camisetas for p in pantalones]
print(f'Total de combinaciones: {len(combinaciones_ropa)}')
for c in combinaciones_ropa:
    print(c)

## 3. Permutaciones: ¿de cuántas formas puedo ordenar algo?

Una **permutación** es una forma de **ordenar** todos los elementos de un conjunto.

Si tengo $n$ elementos distintos, ¿de cuántas formas los puedo ordenar?
- Para la 1ª posición tengo $n$ opciones.
- Para la 2ª, $n-1$ (ya usé una).
- Para la 3ª, $n-2$...
- ...hasta la última, con 1 sola opción.

$$P(n) = n \times (n-1) \times (n-2) \times \dots \times 1 = n!$$

Esto es el **factorial** de $n$, que ya conoces de programación.

In [ ]:
import math
from itertools import permutations

letras = ['A', 'B', 'C']
todas_las_ordenes = list(permutations(letras))

print(f'Permutaciones de {letras}: {len(todas_las_ordenes)}')
print(f'3! = {math.factorial(3)}')
print(todas_las_ordenes)

## 4. Variaciones: ordenar solo una parte

¿Y si no quiero ordenar TODOS los elementos, sino solo elegir y ordenar $k$ de ellos?

Por ejemplo: de 10 corredores, ¿de cuántas formas se puede repartir el podio (oro, plata, bronce)?
Aquí **el orden importa** (no es lo mismo ser 1º que 3º), y solo elegimos 3 de los 10.

$$V(n, k) = n \times (n-1) \times \dots \times (n-k+1) = \dfrac{n!}{(n-k)!}$$

In [ ]:
n, k = 10, 3
variaciones = math.factorial(n) // math.factorial(n - k)
print(f'Formas de repartir el podio entre 10 corredores: {variaciones}')

# Comprobación con itertools.permutations, que ya soporta 'elegir k de n'
corredores = list(range(1, 11))
podios = list(permutations(corredores, 3))
print(f'Comprobación por fuerza bruta: {len(podios)}')

## 5. Combinaciones: cuando el orden NO importa

Volvamos al ejemplo del equipo de 3 personas de un grupo de 10.

A diferencia del podio, aquí **el orden no importa**: el equipo {A, B, C} es el mismo que {C, B, A}.

Como cada equipo de 3 personas se puede ordenar de $3! = 6$ formas distintas, la lista de variaciones
$V(n,k)$ **cuenta cada equipo 6 veces de más**. Para corregirlo, dividimos entre $k!$:

$$C(n, k) = \binom{n}{k} = \dfrac{V(n,k)}{k!} = \dfrac{n!}{k! \,(n-k)!}$$

Este número, $\binom{n}{k}$ (se lee "n sobre k" o "n elige k"), es el famoso **número combinatorio**
o **coeficiente binomial**.

In [ ]:
def combinatorio_factorial(n, k):
    """Calcula C(n, k) usando la fórmula directa con factoriales."""
    return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))

print(f'C(10, 3) = {combinatorio_factorial(10, 3)}')  # Debe coincidir con nuestro ejemplo inicial
print(f'C(10, 3) usando itertools: {len(list(combinations(range(10), 3)))}')

## 6. Otra forma de verlo: la definición recursiva

Los números combinatorios también se pueden definir de forma **recursiva**, pensando así:

Para elegir $k$ elementos de un conjunto de $n$, fíjate en un elemento concreto (el último, por ejemplo).
O bien:
- **lo incluyes** en el grupo → te quedan $k-1$ por elegir entre los $n-1$ restantes → $\binom{n-1}{k-1}$
- **no lo incluyes** → sigues eligiendo $k$ entre los $n-1$ restantes → $\binom{n-1}{k}$

$$\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k}$$

con los casos base $\binom{n}{0} = \binom{n}{n} = 1$.

Esta relación es la base del **Triángulo de Pascal** y de una elegante función recursiva en Python.

In [ ]:
def combinatorio_recursivo(n, k):
    """Calcula C(n, k) usando la relación de Pascal."""
    if k == 0 or k == n:
        return 1
    return combinatorio_recursivo(n - 1, k - 1) + combinatorio_recursivo(n - 1, k)

print(f'C(10, 3) recursivo = {combinatorio_recursivo(10, 3)}')

# Comparamos las tres implementaciones
for n, k in [(5, 2), (10, 3), (6, 0), (6, 6)]:
    a = combinatorio_factorial(n, k)
    b = combinatorio_recursivo(n, k)
    c = math.comb(n, k)  # la versión ya incluida en Python desde 3.8
    print(f'C({n},{k}): factorial={a}, recursivo={b}, math.comb={c}  -> {"OK" if a==b==c else "ERROR"}')

⚠️ **Nota de rendimiento:** la versión recursiva es muy elegante pero **ineficiente**: recalcula los mismos
subproblemas una y otra vez (tiene complejidad exponencial). En clases posteriores de algoritmia verás cómo
solucionar esto con **memoización** o **programación dinámica**. Aquí tienes un adelanto:

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def combinatorio_memo(n, k):
    if k == 0 or k == n:
        return 1
    return combinatorio_memo(n - 1, k - 1) + combinatorio_memo(n - 1, k)

import time

n, k = 28, 14

start = time.perf_counter()
resultado_memo = combinatorio_memo(n, k)
t_memo = time.perf_counter() - start

start = time.perf_counter()
resultado_naive = combinatorio_recursivo(n, k)
t_naive = time.perf_counter() - start

print(f'C({n},{k}) = {resultado_memo}')
print(f'Con memoización: {t_memo:.6f} s')
print(f'Sin memoización:  {t_naive:.6f} s')
print(f'Diferencia: ~{t_naive/t_memo:.0f}x más lento sin memoización')

## 7. El Triángulo de Pascal

Si organizamos $\binom{n}{k}$ para $n = 0, 1, 2, \dots$ y $k = 0, \dots, n$ en filas, obtenemos el famoso
**Triángulo de Pascal**, donde cada número es la suma de los dos que tiene encima (¡justo la relación recursiva de antes!).

In [ ]:
def triangulo_pascal(filas):
    for n in range(filas):
        fila = [math.comb(n, k) for k in range(n + 1)]
        # Centramos visualmente el triángulo
        print(' '.join(f'{v:>4}' for v in fila).center(filas * 6))

triangulo_pascal(10)

In [ ]:
# Visualización gráfica del triángulo con matplotlib
import matplotlib.pyplot as plt

filas = 12
fig, ax = plt.subplots(figsize=(9, 6))

for n in range(filas):
    for k in range(n + 1):
        valor = math.comb(n, k)
        x = k - n / 2
        y = -n
        ax.text(x, y, str(valor), ha='center', va='center', fontsize=9)

ax.set_xlim(-filas/2 - 1, filas/2 + 1)
ax.set_ylim(-filas, 1)
ax.axis('off')
ax.set_title('Triángulo de Pascal: cada número es C(n, k)')
plt.tight_layout()
plt.show()

## 8. ¿Por qué le importa esto a un/a estudiante de Computer Science?

Los números combinatorios aparecen constantemente en informática:

- **Análisis de algoritmos**: contar el número de subconjuntos, particiones o caminos posibles.
- **Probabilidad y estadística**: la distribución binomial usa $\binom{n}{k}$ directamente.
- **Estructuras de datos**: número de árboles binarios distintos (números de Catalan, relacionados con $\binom{2n}{n}$).
- **Criptografía y seguridad**: estimar el tamaño de un espacio de claves o combinaciones posibles.
- **Bases de datos**: cuántas formas hay de elegir columnas para un índice compuesto, etc.

Un ejemplo muy directo: **el número de subconjuntos posibles de un conjunto de $n$ elementos es $2^n$**,
y se puede demostrar que $2^n = \sum_{k=0}^{n} \binom{n}{k}$ (¡la suma de una fila entera del triángulo de Pascal!).

In [ ]:
n = 5
suma_fila = sum(math.comb(n, k) for k in range(n + 1))
print(f'Suma de la fila n={n} del triángulo de Pascal: {suma_fila}')
print(f'2^{n} = {2**n}')
print('¿Coinciden?', suma_fila == 2**n)

## 9. Ejercicios para practicar

Intenta resolver estos ejercicios en las celdas de abajo antes de mirar la solución (usa `math.comb` para comprobar).

1. Una clase tiene 25 estudiantes. ¿De cuántas formas se pueden elegir 4 para un proyecto en grupo?
2. ¿Cuántas contraseñas de 4 letras distintas (sin repetir) se pueden formar con las 26 letras del abecedario? (pista: esto es una **variación**, no una combinación)
3. Escribe una función `combinatorio_iterativo(n, k)` que calcule $\binom{n}{k}$ **sin usar `math.factorial` ni `math.comb`**, usando solo un bucle `for` (pista: simplifica la fórmula multiplicando y dividiendo en el mismo bucle para evitar números enormes).
4. Usa tu función del ejercicio 3 para comprobar que $\binom{n}{k} = \binom{n}{n-k}$ para varios valores de $n$ y $k$.

In [ ]:
# Ejercicio 1: tu código aquí



In [ ]:
# Ejercicio 2: tu código aquí



In [ ]:
# Ejercicio 3: tu código aquí
def combinatorio_iterativo(n, k):
    pass  # completa la función


---
### Soluciones (¡intenta resolverlo antes de mirar!)

In [ ]:
# Solución 1
sol1 = math.comb(25, 4)
print(f'Ejercicio 1: {sol1} grupos posibles')

# Solución 2 (variación, no combinación: el orden importa en una contraseña)
sol2 = math.factorial(26) // math.factorial(26 - 4)
print(f'Ejercicio 2: {sol2} contraseñas posibles')

# Solución 3
def combinatorio_iterativo(n, k):
    k = min(k, n - k)  # aprovechamos la simetría para menos iteraciones
    resultado = 1
    for i in range(k):
        resultado = resultado * (n - i) // (i + 1)
    return resultado

print(f'Ejercicio 3: C(10,3) = {combinatorio_iterativo(10, 3)} (esperado 120)')

# Solución 4: simetría
for n, k in [(10, 3), (8, 2), (15, 6)]:
    izq, der = combinatorio_iterativo(n, k), combinatorio_iterativo(n, n - k)
    print(f'C({n},{k})={izq}  ==  C({n},{n-k})={der}  ->  {izq == der}')

## Resumen

| Concepto | ¿Importa el orden? | ¿Se usan todos los elementos? | Fórmula |
|---|---|---|---|
| Permutación | Sí | Sí | $n!$ |
| Variación | Sí | No (se eligen $k$) | $\dfrac{n!}{(n-k)!}$ |
| Combinación | No | No (se eligen $k$) | $\dbinom{n}{k} = \dfrac{n!}{k!(n-k)!}$ |

El **número combinatorio** $\binom{n}{k}$ responde a la pregunta: *¿de cuántas formas puedo elegir $k$ elementos
de un total de $n$, sin importar el orden?* Es una herramienta fundamental que volverás a encontrar en probabilidad,
análisis de algoritmos y estructuras de datos a lo largo de la carrera. 🎓